In [13]:
from pathlib import Path

import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
# torch (original name before being moved to Python and becoming PyTorch) used to 
# create tensors and store all numerical values, including raw data, weights and biases
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split # importing also random_split for validation
# torch.nn is to make the weight and bias tensors part of the neural network
import torch.nn as nn
# torch.nn.functional is used to import the activation functions
import torch.nn.functional as F


In [14]:

"""
README FIRST

The below code is a template for the solution. You can change the code according
to your preferences, but the testing function has to save the output of your 
model on the test data as it does in this template. This output must be submitted.

Replace the dummy code with your own code in the TODO sections.

We also encourage you to use tensorboard or wandb to log the training process
and the performance of your model. This will help you to debug your model and
to understand how it is performing. But the template does not include this
functionality.
Link for wandb:
https://docs.wandb.ai/quickstart/
Link for tensorboard: 
https://pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html
"""

# The device is automatically set to GPU if available, otherwise CPU
# If you want to force the device to CPU, you can change the line to
# device = torch.device("cpu")

# If you have a Mac consult the following link:
# https://pytorch.org/docs/stable/notes/mps.html

# It is important that your model and all data are on the same device.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()


True

In [15]:
# some playing around with tensors just to see how they work a bit
shape = (2,3,2)
tensor = torch.ones(shape)
print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
tensor = tensor.to('cuda')
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([2, 3, 2])
Datatype of tensor: torch.float32
Device tensor is stored on: cuda:0


In [16]:
# just checking that  file is in the right folder
import os
os.listdir()


['20260421_attempt1_from_template_solution.py',
 '20260425_IML_Task3_Attempt1.ipynb',
 'best_inpainting_model.pth',
 'submit_this_test_data_output.npz',
 'test_image_output',
 'train_image_output']

In [17]:

train_data = np.load("../train_data.npz")["data"]
train_data


array([[[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]],


       [[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]],


       [[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]],


       ...,


       [[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         ...,
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0]]],


       [[[0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 0],
         [0, 0, 0, ..., 0, 0, 

In [18]:

def load_data(**kwargs):
    """
    Get the training and test data. The data files are assumed to be in the
    same directory as this script.

    Args:
    - kwargs: Additional arguments that you might find useful - not necessary

    Returns:
    - train_data_input: Tensor[N_train_samples, C, H, W]
    - train_data_label: Tensor[N_train_samples, C, H, W]
    - test_data_input: Tensor[N_test_samples, C, H, W]
    where N_train_samples is the number of training samples, N_test_samples is
    the number of test samples, C is the number of channels (1 for grayscale),
    H is the height of the image, and W is the width of the image.
    """
    # Load the training data
    train_data = np.load("../train_data.npz")["data"]

    # Make the training data a tensor
    train_data = torch.tensor(train_data, dtype=torch.float32) / 255.0

    # Load the test data
    test_data_input = np.load("../test_data.npz")["data"]

    # Make the test data a tensor
    test_data_input = torch.tensor(test_data_input, dtype=torch.float32) / 255.0

    ########################################
    # TODO: Given the original training images, create the input images and the
    # label images to train your model. 
    # Replace the two placholder lines below (which currently just copy the
    # training data) with your own implementation.

    # as per the test images seen below, a square mask of 64 pixels from x = 10, y = 10 to x = 18, y = 18 will be applied
    train_data_label = train_data.clone()
    train_data_input = train_data.clone()
    train_data_input[:, :, 10:18, 10:18] = 0.0

    # Visualize the training data if needed
    # Set to False if you don't want to save the images
    if True:
        # Create the output directory if it doesn't exist
        if not Path("train_image_output").exists():
            Path("train_image_output").mkdir()
        for i in tqdm(range(20), desc="Plotting train images"):
            # Show the training and the target image side by side
            plt.subplot(1, 2, 1)
            plt.imshow(train_data_input[i].squeeze(), cmap="gray")
            plt.title("Training Input")
            plt.subplot(1, 2, 2)
            plt.title("Training Label")
            plt.imshow(train_data_label[i].squeeze(), cmap="gray")

            plt.savefig(f"train_image_output/image_{i}.png")
            plt.close()

# having a look at the Test images to understand if they have been masked and denoised, conclusion: masked from 10 to 18 pixels x and y axis
        if True:
        # Create the output directory if it doesn't exist
            if not Path("test_image_output").exists():
                Path("test_image_output").mkdir()
            for i in tqdm(range(20), desc="Plotting test images"):
                # Show the training and the target image side by side
                plt.subplot(1, 2, 1)
                plt.imshow(test_data_input[i].squeeze(), cmap="gray")
                plt.title("Test Input")

                plt.savefig(f"test_image_output/image_{i}.png")
                plt.close()

    return train_data_input, train_data_label, test_data_input



In [19]:
load_data()

Plotting test images: 100%|██████████| 20/20 [00:02<00:00,  6.69it/s]


(tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]],
 
 
         [[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]],
 
 
         [[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]],
 
 
         ...,
 
 
         [[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ..

In [ ]:

def training(train_data_input, train_data_label, **kwargs):
    """
    Train the model. Fill in the details of the data loader, the loss function,
    the optimizer, and the training loop.

    Args:
    - train_data_input: Tensor[N_train_samples, C, H, W]
    - train_data_label: Tensor[N_train_samples, C, H, W]
    - kwargs: Additional arguments that you might find useful - not necessary

    Returns:
    - model: torch.nn.Module
    """
    model = Model()
    model.train()
    model.to(device)

    # TODO: Dummy criterion - change this to the correct loss function
    # https://pytorch.org/docs/stable/nn.html#loss-functions
    criterion = lambda x, y: torch.mean((x))
    criterion = nn.L1Loss() # would be nn.CrossEntropyLoss for integer labels
    # TODO: Dummy optimizer - change this to a more suitable optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.001,) 
    # optimiser could be optim.SGD(....,momentum = 0.9) as per PyTorch tutorial and Medium article for integer values

    # TODO: Correctly setup the dataloader - the below is just a placeholder
    # Also consider that you might not want to use the entire dataset for
    # training alone
    # (batch_size needs to be changed)
    batch_size = 4 # PyTorch tutorial has 4
    dataset = TensorDataset(train_data_input, train_data_label)
    # Consider the shuffle parameter and other parameters of the DataLoader
    # class (see
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
    train_size = int(0.8 * len(dataset)) # 80% = 16 images
    val_size = len(dataset) - train_size # 20% = 4 images
    
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Training loop
    # TODO: Modify the training loop in case you need to
    # splitting data into training and validation, just to try to avoid overfitting
    # TODO: The value of n_epochs is just a placeholder and likely needs to be
    # changed

    best_val_loss = float('inf')
    patience = 10 # how many epochs to wait for an improvement
    patience_counter = 0   

    n_epochs = 100

    for epoch in range(n_epochs):
        model.train() # just to be sure
        running_train_loss = 0.0

        for x, y in tqdm(
            train_loader, desc=f"Training Epoch {epoch}", leave=False
        ):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * x.size(0)

        avg_train_loss = running_train_loss / len(train_dataset)

        model.eval() # putting model in eval mode (freezes dropout/batchnorm)
        running_val_loss = 0.0
        
        with torch.no_grad(): # not tracking gradients for validation
            for val_x, val_y in val_loader:
                val_x, val_y = val_x.to(device), val_y.to(device)
                val_output = model(val_x)
                val_loss = criterion(val_output, val_y)
                running_val_loss += val_loss.item() * val_x.size(0)
                
        avg_val_loss = running_val_loss / len(val_dataset)

        print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

# implement a patience_counter that makes the loop stop if in 10 epochs the val loss does not decrease
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0 # Reset counter because we got a new best!
            
            # Pro-tip: Save the best model weights so you keep the best version!
            torch.save(model.state_dict(), "best_inpainting_model.pth")
        else:
            patience_counter += 1 # We didn't improve, add 1 to the strike counter
            print(f"No improvement in Val Loss. Patience: {patience_counter}/{patience}")

        # If we strike out, stop the loop!
        if patience_counter >= patience:
            print(f"Early stopping triggered at Epoch {epoch}!")
            # Load the best weights back into the model before returning it
            model.load_state_dict(torch.load("best_inpainting_model.pth"))
            break

    return model



In [21]:

# TODO: define a model. Here, a basic MLP model is defined. You can completely
# change this model - and are encouraged to do so.
# inspired by example found on Kaggle
   

class Model(nn.Module):
    """
    Convolutional Autoencoder for Image Inpainting.
    """

    def __init__(self):
        super().__init__()
        
        # 1. first an encoder shrinks the image and extract the context around the mask
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1), # 28x28 -> 14x14. Padding is to ensure corners are also searched
            # kernel_size = 3, stride = 2 and padding = 1, is good for even-pixel images because the convolutions travels the entire image well
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1), # 14x14 -> 7x7
            nn.ReLU()
        )
        
        # 2. DECODER: Expands the feature map back into a full 28x28 image
        self.decoder = nn.Sequential(
            # ConvTranspose2d does the exact opposite of Conv2d (it scales UP)
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1), # 7x7 -> 14x14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1), # 14x14 -> 28x28
            nn.Sigmoid() # Keeps the final pixel values strictly between 0.0 and 1.0
        )

    def forward(self, x):
        """
        The forward pass of the model.
        input: x: torch.Tensor [Batch, 1, 28, 28] (The masked image)
        output: x: torch.Tensor [Batch, 1, 28, 28] (The repaired image)
        """
        # Step 1: Compress the image to learn the context
        x = self.encoder(x)
        
        # Step 2: Decompress the image to fill in the mask
        x = self.decoder(x)
        
        return x



In [22]:

def testing(model, test_data_input):
    """
    Uses your model to predict the ouputs for the test data. Saves the outputs
    as a binary file. This file needs to be submitted. This function does not
    need to be modified except for setting the batch_size value. If you choose
    to modify it otherwise, please ensure that the generating and saving of the
    output data is not modified.

    Args:
    - model: torch.nn.Module
    - test_data_input: Tensor
    """
    model.eval()
    model.to(device)

    with torch.no_grad():
        test_data_input = test_data_input.to(device)
        # Predict the output batch-wise to avoid memory issues
        test_data_output = []
        # TODO: You can increase or decrease this batch size depending on your
        # memory requirements of your computer / model
        # This will not affect the performance of the model and your score
        batch_size = 64
        for i in tqdm(
            range(0, test_data_input.shape[0], batch_size),
            desc="Predicting test output",
        ):
            output = model(test_data_input[i : i + batch_size])
            test_data_output.append(output.cpu())
        test_data_output = torch.cat(test_data_output)

    # Ensure the output has the correct shape
    assert test_data_output.shape == test_data_input.shape, (
        f"Expected shape {test_data_input.shape}, but got "
        f"{test_data_output.shape}."
        "Please ensure the output has the correct shape."
        "Without the correct shape, the submission cannot be evaluated and "
        "will hence not be valid."
    )

    # Save the output
    test_data_output = test_data_output.numpy()
    # Ensure all values are in the range [0, 255]
    test_data_output = test_data_output * 255.0 # added because of comment above and the Sigmoid function is used at the end of model
    save_data_clipped = np.clip(test_data_output, 0, 255)
    # Convert to uint8
    # Ensure your model outputs values in the [0, 255] range before this step! If you normalized your data to [0, 1], you must multiply by 255 before saving.
    save_data_uint8 = save_data_clipped.astype(np.uint8)
    # Loss is only computed on the masked area - so set the rest to 0 to save
    # space
    save_data = np.zeros_like(save_data_uint8)
    save_data[:, :, 10:18, 10:18] = save_data_uint8[:, :, 10:18, 10:18]

    np.savez_compressed(
        "submit_this_test_data_output.npz", data=save_data)

    # You can plot the output if you want
    # Set to False if you don't want to save the images
    if True:
        # Create the output directory if it doesn't exist
        if not Path("test_image_output").exists():
            Path("test_image_output").mkdir()
        for i in tqdm(range(20), desc="Plotting test images"):
            # Show the training and the target image side by side
            plt.subplot(1, 2, 1)
            plt.title("Test Input")
            plt.imshow(test_data_input[i].squeeze().cpu().numpy(), cmap="gray")
            plt.subplot(1, 2, 2)
            plt.imshow(test_data_output[i].squeeze(), cmap="gray")
            plt.title("Test Output")

            plt.savefig(f"test_image_output/image_{i}.png")
            plt.close()


In [23]:


def main():
    seed = 0
    # Reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

    # You don't need to change the code below
    # Load the data
    train_data_input, train_data_label, test_data_input = load_data()
    # Train the model
    model = training(train_data_input, train_data_label)

    # Test the model (this also generates the submission file)
    # The name of the submission file is submit_this_test_data_output.npz
    testing(model, test_data_input)

    return None


if __name__ == "__main__":
    main()


Plotting test images: 100%|██████████| 20/20 [00:02<00:00,  6.82it/s]


Epoch 0 | Train Loss: 0.013931 | Val Loss: 0.010337


Epoch 1 | Train Loss: 0.009931 | Val Loss: 0.009623


Epoch 2 | Train Loss: 0.009382 | Val Loss: 0.009220


Epoch 3 | Train Loss: 0.009127 | Val Loss: 0.008994


...

Epoch 60 | Train Loss: 0.007956 | Val Loss: 0.008150
No improvement in Val Loss. Patience: 7/10


Epoch 61 | Train Loss: 0.007954 | Val Loss: 0.008012
No improvement in Val Loss. Patience: 8/10


Epoch 62 | Train Loss: 0.007948 | Val Loss: 0.007941
No improvement in Val Loss. Patience: 9/10


Epoch 63 | Train Loss: 0.007942 | Val Loss: 0.007938
No improvement in Val Loss. Patience: 10/10
Early stopping triggered at Epoch 63!


Plotting test images: 100%|██████████| 20/20 [00:06<00:00,  3.15it/s]
